# Почему упал Фин.рез: полная проверка

Тетрадка **только читает** DRP (никаких DELETE / UPDATE / DROP). Источник — та же таблица, что у дашборда: `sbx_da.tmp_shestopalov_acq_fin_version_test`.

Формулы DAG (`acq_slice_load.sql`), по ним всё и раскладывается:
- **ЧОД** = комиссия с операций (`commission_from_ops`) + ежемесячная плата (`commission_monthly`) + межбанковская комиссия (`int_component`, отрицательная)
- **Фин.рез** = ЧОД − АУР (`aur` = точки с терминалами × 1926 ₽) − амортизация терминалов (`amortization`)
- **Маржа на операциях** = комиссия с операций + межбанк

| Раздел | На какой вопрос отвечает |
|---|---|
| 2 | Правильно ли посчитаны ЧОД и Фин.рез в каждой строке |
| 3 | Не потерялись ли данные (договоры, операции) и подтверждает ли падение оборота независимая таблица MCC |
| 4 | P&L по месяцам и ставки |
| 5 | На сколько млн повлиял каждый фактор (оборот, ставки, ежемесячная плата, расходы) — сумма факторов = факт |
| 6 | Какие клиенты дали падение: ушли, сократили оборот, топ-клиенты |
| 7 | Тарифы: падение ставки из-за цен или из-за сдвига оборота в дешёвые тарифы |
| 8 | Филиалы: падение везде или в нескольких |
| 9 | Почему август был выше: разовая ежемесячная плата |
| 10 | Платёжные системы и ставка межбанка |
| 11 | Сверка с Excel-отчётами коллег (`Data/09_Сентябрь_2026.xlsx` и т. д.): итоги по месяцам и по договорам, видно ли падение в их отчётах |
| 12 | Итоговая справка + Excel со всеми таблицами |

**Запуск:** Kernel → Restart Kernel and Run All Cells. Спросит логин и пароль DRP (не больше 2 попыток). Работает 1–3 минуты.

Сравниваемые месяцы меняются в первой ячейке (`BASE_MONTH`, `CMP_MONTH`). Открытый месяц (ноябрь) в выводы не берите: ежемесячная плата списывается в конце месяца.

## 1. Настройки и подключение

In [ ]:
from getpass import getpass
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from rail_connectors.connection import connect

pd.set_option('display.max_columns', 60)
pd.set_option('display.max_rows', 200)
pd.set_option('display.width', 250)

TABLE = 'sbx_da.tmp_shestopalov_acq_fin_version_test'
MCC_TABLE = 'sbx_da.tmp_shestopalov_acq_mcc_month'
CARD_TABLE = 'sbx_da.tmp_shestopalov_acq_card_type_month'

BASE_MONTH = '2026-08'
CMP_MONTH = '2026-10'
EXTRA_PAIRS = [('2026-07', '2026-09'), ('2026-08', '2026-09'), ('2026-07', '2026-10')]
ANOMALY_MONTH = '2026-08'
OPEN_MONTH = '2026-11'
TOP_N = 20
OUT_DIR = Path('Data/fin_drop_analysis')
OUT_DIR.mkdir(parents=True, exist_ok=True)
DRP_USER_DEFAULT = 'Shestopalov-VYur'
MLN = 1e6

if 'drp' not in globals() or drp is None:
    drp_user = input('DRP user: ').strip() or DRP_USER_DEFAULT
    drp = connect(to='DRP', user_params={'user_name': drp_user, 'password': getpass('DRP password: ')})


def fetch(sql):
    with drp:
        res = drp.fetch(sql)
    return pd.DataFrame() if res is None else res


def month_sum(table, num_cols, txt_cols=()):
    cols = ['report_month'] + list(txt_cols) + list(num_cols)
    t = fetch('SELECT ' + ', '.join(cols) + ' FROM ' + table)
    t.columns = [str(c).strip().lower() for c in t.columns]
    t['m'] = t['report_month'].astype(str).str.strip().str[:7]
    for c in txt_cols:
        t[c] = t[c].astype(str).str.strip()
    for c in num_cols:
        t[c] = pd.to_numeric(t[c], errors='coerce').fillna(0)
    return t


print('Таблица:', TABLE)
print('Главное сравнение:', BASE_MONTH, '->', CMP_MONTH, '| доп. пары:', EXTRA_PAIRS, '| открытый месяц:', OPEN_MONTH)

## Загрузка данных

Берём все месяцы одной выборкой (~75 тыс. строк). Пустые числа считаем нулём, но сколько их было — сохраняем для раздела 3.

In [ ]:
NUM_COLS = ['trx_cnt', 'trx_sum', 'commission_from_ops', 'commission_monthly', 'int_component',
            'commission_total', 'chod', 'aur', 'amortization', 'fin_result',
            'retl_cnt', 'retl_with_term_cnt', 'term_cnt', 'active_terms']
TXT_COLS = ['report_month', 'inn', 'company_name', 'agr_id', 'filial_rf', 'tariff_short', 'tariff_name',
            'commission_monthly_source']


def clean_txt(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return None
    s = str(x).strip()
    return None if s in ('', '<NULL>', 'None', 'nan', 'NaN') else s


cols_sql = ', '.join('CAST(' + c + ' AS TEXT) AS ' + c for c in TXT_COLS + NUM_COLS)
raw = fetch('SELECT ' + cols_sql + ' FROM ' + TABLE)
raw.columns = [str(c).strip().lower() for c in raw.columns]

df = raw.copy()
for c in TXT_COLS:
    df[c] = df[c].map(clean_txt)
for c in NUM_COLS:
    df[c] = pd.to_numeric(df[c], errors='coerce')
df_isna = df[NUM_COLS].isna()
df[NUM_COLS] = df[NUM_COLS].fillna(0.0)

df['m'] = df['report_month'].str[:7]
df['client'] = df['inn'].where(df['inn'].notna(), 'agr:' + df['agr_id'].fillna('?').astype(str))
df['filial'] = df['filial_rf'].fillna('ЦРМБ / нет информации')
df['tariff'] = df['tariff_short'].fillna('Не сегментирован')
df['monthly_src'] = df['commission_monthly_source'].fillna('нет')

months = sorted(df['m'].dropna().unique().tolist())
need = sorted({BASE_MONTH, CMP_MONTH, ANOMALY_MONTH} | {x for p in EXTRA_PAIRS for x in p})
lost = [x for x in need if x not in months]
assert not lost, 'Нет месяцев ' + str(lost) + ' в ' + TABLE + ': ' + str(months)
CLOSED = [x for x in months if x < OPEN_MONTH]
print('строк:', f'{len(df):,}', '| месяцы:', months)
print('закрытые месяцы (для выводов):', CLOSED)

## 2. Формулы сходятся построчно?

Для каждой строки пересчитываем ЧОД, Фин.рез, АУР и общую комиссию и сравниваем с тем, что лежит в таблице. Допуск 5 копеек (округление).

**Норма:** во всех колонках `*_bad` нули. Тогда цифры на дашборде — это ровно сумма компонентов, и раскладка в разделе 5 корректна.

In [ ]:
aur_calc = np.minimum(np.maximum(df['retl_with_term_cnt'], 0), df['retl_cnt']) * 1926
chk = pd.DataFrame({
    'm': df['m'],
    'chod_diff': (df['commission_from_ops'] + df['commission_monthly'] + df['int_component'] - df['chod']).abs(),
    'fin_diff': (df['chod'] - df['aur'] - df['amortization'] - df['fin_result']).abs(),
    'aur_diff': (aur_calc - df['aur']).abs(),
    'total_diff': (df['commission_from_ops'] + df['commission_monthly'] - df['commission_total']).abs(),
})
TOL = 0.05
formula = chk.groupby('m').agg(
    rows=('chod_diff', 'size'),
    chod_bad=('chod_diff', lambda s: int((s > TOL).sum())),
    fin_bad=('fin_diff', lambda s: int((s > TOL).sum())),
    aur_bad=('aur_diff', lambda s: int((s > TOL).sum())),
    total_bad=('total_diff', lambda s: int((s > TOL).sum())),
    chod_diff_rub=('chod_diff', 'sum'),
    fin_diff_rub=('fin_diff', 'sum'),
)
display(formula.round(2))
BAD_COLS = ['chod_bad', 'fin_bad', 'aur_bad', 'total_bad']
bad_months = formula.index[formula[BAD_COLS].sum(axis=1) > 0].tolist()
FORMULA_OK = not [x for x in bad_months if x in need]
print('Месяцы с расхождениями:', bad_months or 'нет')
print('Формулы в сравниваемых месяцах', need, ':', 'СХОДЯТСЯ' if FORMULA_OK else 'ЕСТЬ РАСХОЖДЕНИЯ — пришлите скриншот')

## 3. Полнота данных и независимая сверка оборота

**Первая таблица — полнота по месяцам.** Если бы DAG потерял данные, резко упало бы число договоров / клиентов / договоров с операциями, или выросли бы пустые значения (`null_*`), договоры без филиала, договоры без документа ежемесячной платы (`monthly_doc_missing`).

**Вторая таблица — сверка с таблицей MCC.** Таблица MCC считается **отдельно от DAG**, напрямую из операций в озере. Если оборот в ней падает так же, как в основной таблице (пропорция `ratio_sum` стабильна по месяцам), значит падение оборота **реальное**, а не ошибка расчёта DAG. Оборот MCC немного шире периметра договоров, поэтому пропорция не 1, но она должна быть примерно одинаковой каждый месяц.

In [ ]:
comp = df.groupby('m').agg(
    rows=('agr_id', 'size'),
    agreements=('agr_id', 'nunique'),
    clients=('client', 'nunique'),
    agr_with_trx=('trx_sum', lambda s: int((s > 0).sum())),
    agr_with_monthly=('commission_monthly', lambda s: int((s != 0).sum())),
    trx_cnt_mln=('trx_cnt', lambda s: s.sum() / MLN),
    trx_sum_mln=('trx_sum', lambda s: s.sum() / MLN),
    retl_cnt=('retl_cnt', 'sum'),
    term_cnt=('term_cnt', 'sum'),
)
comp['no_filial'] = df['filial_rf'].isna().groupby(df['m']).sum()
comp['monthly_doc_missing'] = df['monthly_src'].eq('oper_doc_missing').groupby(df['m']).sum()
nulls = df_isna[['trx_sum', 'commission_from_ops', 'int_component', 'chod', 'fin_result']].groupby(df['m']).sum()
nulls.columns = ['null_' + c for c in nulls.columns]
comp = comp.join(nulls)
print('Полнота по месяцам:')
display(comp.round(2))

mcc_raw = month_sum(MCC_TABLE, ['trx_sum', 'trx_cnt'])
mcc_m = mcc_raw.groupby('m')[['trx_sum', 'trx_cnt']].sum() / MLN
mcc_m.columns = ['mcc_trx_sum_mln', 'mcc_trx_cnt_mln']
cross = comp[['trx_sum_mln', 'trx_cnt_mln']].join(mcc_m)
cross['ratio_sum'] = cross['trx_sum_mln'] / cross['mcc_trx_sum_mln']
cross['ratio_cnt'] = cross['trx_cnt_mln'] / cross['mcc_trx_cnt_mln']
cross['fin_trx_sum_vs_prev_pct'] = cross['trx_sum_mln'].pct_change() * 100
cross['mcc_trx_sum_vs_prev_pct'] = cross['mcc_trx_sum_mln'].pct_change() * 100
print('Сверка оборота с независимой таблицей MCC:')
display(cross.round(3))

ref_months = [x for x in cross.index if x < '2026-09' and pd.notna(cross.loc[x, 'ratio_sum'])]
ref_ratio = cross.loc[ref_months, 'ratio_sum'].median()
CROSS_DEV = abs(cross.loc[CMP_MONTH, 'ratio_sum'] / ref_ratio - 1) * 100
AGR_CHG = (comp.loc[CMP_MONTH, 'agreements'] / comp.loc[BASE_MONTH, 'agreements'] - 1) * 100
MCC_TRX_CHG = (cross.loc[CMP_MONTH, 'mcc_trx_sum_mln'] / cross.loc[BASE_MONTH, 'mcc_trx_sum_mln'] - 1) * 100
FIN_TRX_CHG = (cross.loc[CMP_MONTH, 'trx_sum_mln'] / cross.loc[BASE_MONTH, 'trx_sum_mln'] - 1) * 100
print(f'Договоров {BASE_MONTH} -> {CMP_MONTH}: {AGR_CHG:+.1f}%')
print(f'Оборот: основная таблица {FIN_TRX_CHG:+.1f}%, независимая таблица MCC {MCC_TRX_CHG:+.1f}%')
print(f'Пропорция оборота к MCC в {CMP_MONTH} отличается от обычной (медиана янв–авг) на {CROSS_DEV:.1f}%')

## 4. P&L по месяцам и ставки

Все суммы в млн ₽. Ставки — в % от оборота. Межбанк отрицательный (это расход). **Спред** = ставка комиссии + ставка межбанка = сколько банк зарабатывает с каждого рубля оборота до ежемесячной платы и расходов.

ЧОД и Фин.рез здесь должны совпасть с графиком «ЧОД ТЭ + Фин.рез.» на дашборде (при выборе всех филиалов).

In [ ]:
def pnl(frame, by):
    g = frame.groupby(by).agg(
        trx_sum=('trx_sum', 'sum'), trx_cnt=('trx_cnt', 'sum'),
        ops=('commission_from_ops', 'sum'), int_=('int_component', 'sum'),
        monthly=('commission_monthly', 'sum'), chod=('chod', 'sum'), aur=('aur', 'sum'),
        amort=('amortization', 'sum'), fin=('fin_result', 'sum'),
        agreements=('agr_id', 'nunique'), clients=('client', 'nunique'),
    )
    g['margin'] = g['ops'] + g['int_']
    g['ops_pct'] = g['ops'] / g['trx_sum'] * 100
    g['int_pct'] = g['int_'] / g['trx_sum'] * 100
    g['spread_pct'] = g['margin'] / g['trx_sum'] * 100
    g['avg_check'] = g['trx_sum'] / g['trx_cnt']
    return g.replace([np.inf, -np.inf], np.nan)


monthly = pnl(df, 'm')
MONEY = ['trx_sum', 'ops', 'int_', 'margin', 'monthly', 'chod', 'aur', 'amort', 'fin']
show = monthly[MONEY + ['ops_pct', 'int_pct', 'spread_pct', 'avg_check', 'trx_cnt', 'agreements', 'clients']].copy()
show[MONEY] = show[MONEY] / MLN
show['trx_cnt'] = show['trx_cnt'] / MLN
show.columns = ['Оборот', 'Комиссия с операций', 'Межбанк', 'Маржа на операциях', 'Ежемесячная плата',
                'ЧОД', 'АУР', 'Амортизация', 'Фин.рез', 'Ставка комиссии, %', 'Ставка межбанка, %', 'Спред, %',
                'Средний чек, ₽', 'Операций, млн', 'Договоров', 'Клиентов']
display(show.round(3))

h1 = [x for x in CLOSED if x <= '2026-06']
base_avg = monthly.loc[h1, 'trx_sum'].mean()
season = pd.DataFrame({
    'Оборот, млн': monthly['trx_sum'] / MLN,
    'Оборот к среднему янв–июн, %': monthly['trx_sum'] / base_avg * 100,
    'Операций к среднему янв–июн, %': monthly['trx_cnt'] / monthly.loc[h1, 'trx_cnt'].mean() * 100,
    'Средний чек, ₽': monthly['avg_check'],
})
print('Динамика оборота (сезонность):')
display(season.round(1))

## 5. Факторы изменения Фин.реза (главная таблица)

Изменение Фин.реза раскладывается **точно** (сумма факторов = факт, невязка ≈ 0):

1. **Объём оборота** — сколько потеряли бы только из-за меньшего оборота, если бы ставки остались как в базовом месяце.
2. **Ставка комиссии с операций** — эффект изменения средней ставки, которую платят клиенты.
3. **Ставка межбанка** — эффект изменения ставки межбанковской комиссии (минус = расход вырос).
4. **Ежемесячная плата** — разница в фиксированных платежах клиентов.
5. **АУР** и 6. **Амортизация** — изменение расходов (минус = расходы выросли).

Ниже главной таблицы — те же факторы для дополнительных пар месяцев: так видно, что вывод не зависит от выбора «базы» (например, август был необычным).

In [ ]:
FACTORS = ['1. Объём оборота (при ставках базового месяца)', '2. Ставка комиссии с операций',
           '3. Ставка межбанковской комиссии', '4. Ежемесячная плата',
           '5. АУР (точки с терминалами × 1926 ₽)', '6. Амортизация терминалов']


def bridge(m0, m1):
    a, b = monthly.loc[m0], monthly.loc[m1]
    r0, i0 = a.ops / a.trx_sum, a.int_ / a.trx_sum
    r1, i1 = b.ops / b.trx_sum, b.int_ / b.trx_sum
    vals = [
        (b.trx_sum - a.trx_sum) * (r0 + i0),
        b.trx_sum * (r1 - r0),
        b.trx_sum * (i1 - i0),
        b.monthly - a.monthly,
        -(b.aur - a.aur),
        -(b.amort - a.amort),
    ]
    vals = [v / MLN for v in vals]
    fact = (b.fin - a.fin) / MLN
    explained = sum(vals)
    out = pd.DataFrame({
        'Фактор': FACTORS + ['ИТОГО по факторам', 'Факт: изменение Фин.реза', 'Невязка (должна быть ≈ 0)'],
        'Влияние, млн': vals + [explained, fact, explained - fact],
    })
    out['Доля в изменении, %'] = np.where(out.index < len(FACTORS), out['Влияние, млн'] / fact * 100, np.nan)
    return out


a, b = monthly.loc[BASE_MONTH], monthly.loc[CMP_MONTH]
print(f'Фин.рез {BASE_MONTH}: {a.fin / MLN:,.2f} млн -> {CMP_MONTH}: {b.fin / MLN:,.2f} млн')
print(f'Ставка комиссии {a.ops_pct:.3f}% -> {b.ops_pct:.3f}% | ставка межбанка {a.int_pct:.3f}% -> {b.int_pct:.3f}% | спред {a.spread_pct:.3f}% -> {b.spread_pct:.3f}%')
main_bridge = bridge(BASE_MONTH, CMP_MONTH)
display(main_bridge.round(2))

pairs = [(BASE_MONTH, CMP_MONTH)] + [p for p in EXTRA_PAIRS if p != (BASE_MONTH, CMP_MONTH)]
pair_bridges = {p: bridge(*p) for p in pairs}
pairs_tbl = pd.DataFrame({p[0] + ' → ' + p[1]: t.set_index('Фактор')['Влияние, млн'] for p, t in pair_bridges.items()})
print('Те же факторы для разных пар месяцев, млн:')
display(pairs_tbl.round(2))

## 6. Клиенты: кто дал падение

Клиент = ИНН (договоры одного ИНН суммируются). Сравниваем базовый и сравниваемый месяц.

**Первая таблица — группы клиентов.** Сумма по группам = общее изменение Фин.реза.
- **Перестал проводить операции** — в базовом месяце был оборот, теперь нет (ушёл / приостановил).
- **Оборот снизился** / **вырос** — работает в обоих месяцах.

Если падение дали в основном «Оборот снизился» у сотен клиентов — это общий спад (сезонность, рынок). Если несколько клиентов — это конкретные клиенты, их список ниже.

**Топ-списки** — в тыс. ₽: клиенты с наибольшим падением и ростом Фин.реза.

In [ ]:
def by_client(m):
    x = df[df['m'] == m]
    g = x.groupby('client').agg(
        company=('company_name', 'last'), filial=('filial', 'last'), tariff=('tariff', 'last'),
        trx_sum=('trx_sum', 'sum'), ops=('commission_from_ops', 'sum'), int_=('int_component', 'sum'),
        monthly=('commission_monthly', 'sum'), aur=('aur', 'sum'), amort=('amortization', 'sum'),
        fin=('fin_result', 'sum'), agreements=('agr_id', 'nunique'),
    )
    g['margin'] = g['ops'] + g['int_']
    return g


def client_bridge(m0, m1):
    j = by_client(m0).join(by_client(m1), how='outer', lsuffix='_0', rsuffix='_1')
    for c in ['trx_sum', 'ops', 'int_', 'monthly', 'aur', 'amort', 'fin', 'margin', 'agreements']:
        j[c + '_0'] = j[c + '_0'].fillna(0)
        j[c + '_1'] = j[c + '_1'].fillna(0)
    for c in ['company', 'filial', 'tariff']:
        j[c] = j[c + '_1'].where(j[c + '_1'].notna(), j[c + '_0'])
    j['d_trx'] = j['trx_sum_1'] - j['trx_sum_0']
    j['d_margin'] = j['margin_1'] - j['margin_0']
    j['d_monthly'] = j['monthly_1'] - j['monthly_0']
    j['d_costs'] = -((j['aur_1'] - j['aur_0']) + (j['amort_1'] - j['amort_0']))
    j['d_fin'] = j['fin_1'] - j['fin_0']
    has0, has1 = j['trx_sum_0'] > 0, j['trx_sum_1'] > 0
    j['status'] = np.select(
        [has0 & ~has1, ~has0 & has1, has0 & has1 & (j['d_trx'] < 0), has0 & has1],
        ['Перестал проводить операции', 'Начал проводить операции', 'Оборот снизился', 'Оборот вырос или не изменился'],
        default='Без оборота в обоих месяцах',
    )
    return j


cb = client_bridge(BASE_MONTH, CMP_MONTH)
cb_status = cb.groupby('status').agg(
    clients=('d_fin', 'size'), d_trx=('d_trx', 'sum'), d_margin=('d_margin', 'sum'),
    d_monthly=('d_monthly', 'sum'), d_costs=('d_costs', 'sum'), d_fin=('d_fin', 'sum'),
)
money = ['d_trx', 'd_margin', 'd_monthly', 'd_costs', 'd_fin']
cb_status[money] = cb_status[money] / MLN
cb_status.loc['ИТОГО'] = cb_status.sum()
cb_status.columns = ['Клиентов', 'Изм. оборота, млн', 'Изм. маржи на операциях, млн', 'Изм. ежемес. платы, млн',
                     'Изм. расходов, млн', 'Изм. Фин.реза, млн']
print(f'Группы клиентов, {BASE_MONTH} -> {CMP_MONTH}:')
display(cb_status.round(2))
print(f'Сверка: сумма по клиентам {cb.d_fin.sum() / MLN:,.2f} млн = факт {(b.fin - a.fin) / MLN:,.2f} млн')

both = cb[(cb['trx_sum_0'] > 0) & (cb['trx_sum_1'] > 0)]
SHARE_DOWN = (both['d_trx'] < 0).mean() * 100
print(f'Из клиентов, работающих в обоих месяцах ({len(both):,}), оборот снизился у {SHARE_DOWN:.0f}%')

TOP_COLS = ['company', 'filial', 'tariff', 'status', 'trx_sum_0', 'trx_sum_1', 'margin_0', 'margin_1',
            'monthly_0', 'monthly_1', 'd_costs', 'd_fin']
TOP_NAMES = ['Клиент', 'Филиал', 'Тариф', 'Группа', 'Оборот ' + BASE_MONTH, 'Оборот ' + CMP_MONTH,
             'Маржа ' + BASE_MONTH, 'Маржа ' + CMP_MONTH, 'Ежемес. ' + BASE_MONTH, 'Ежемес. ' + CMP_MONTH,
             'Изм. расходов', 'Изм. Фин.реза']


def fmt_top(t):
    t = t[TOP_COLS].copy()
    t[TOP_COLS[4:]] = t[TOP_COLS[4:]] / 1e3
    t.columns = TOP_NAMES
    t.index.name = 'ИНН'
    return t.round(1)


cb_sorted = cb.sort_values('d_fin')
cb_top_down = fmt_top(cb_sorted.head(TOP_N))
cb_top_up = fmt_top(cb_sorted.tail(10).iloc[::-1])
TOP_DOWN_SUM = cb_sorted.head(TOP_N)['d_fin'].sum() / MLN
TOP10_DOWN_SUM = cb_sorted.head(10)['d_fin'].sum() / MLN
print(f'Топ-{TOP_N} клиентов по падению Фин.реза (тыс. ₽). Вместе: {TOP_DOWN_SUM:+,.2f} млн, топ-10: {TOP10_DOWN_SUM:+,.2f} млн')
display(cb_top_down)
print('Топ-10 клиентов по росту Фин.реза (тыс. ₽):')
display(cb_top_up)

cb_top_trx = fmt_top(cb.sort_values('d_trx').head(TOP_N))
TOP_TRX_SUM = cb.sort_values('d_trx').head(TOP_N)['d_trx'].sum() / MLN
print(f'Топ-{TOP_N} клиентов по падению оборота: вместе {TOP_TRX_SUM:+,.1f} млн из {cb.d_trx.sum() / MLN:+,.1f} млн общего изменения')
display(cb_top_trx)

## 7. Тарифы: цены или структура

Изменение **маржи на операциях** раскладывается на три части:
- **Объём оборота** — меньше оборота при той же средней марже;
- **Структура** — оборот перетёк в тарифы с меньшей маржой (например, в акционные / индивидуальные);
- **Ставки внутри групп** — внутри одних и тех же тарифов маржа на рубль стала меньше (снизили ставки клиентам или вырос межбанк).

Ниже — доли оборота и спред по тарифам за все месяцы, чтобы видеть тренд.

In [ ]:
def mix_bridge(m0, m1, dim):
    g = pnl(df[df['m'].isin([m0, m1])], ['m', dim])
    ga = g.xs(m0, level='m')
    gb = g.xs(m1, level='m')
    idx = ga.index.union(gb.index)
    ga = ga.reindex(idx).fillna(0)
    gb = gb.reindex(idx).fillna(0)
    t0, t1 = ga['trx_sum'].sum(), gb['trx_sum'].sum()
    s0t = ga['margin'].sum() / t0
    s0 = (ga['margin'] / ga['trx_sum'].where(ga['trx_sum'] > 0)).fillna(s0t)
    s1 = (gb['margin'] / gb['trx_sum'].where(gb['trx_sum'] > 0)).fillna(0)
    eff = {
        'Объём оборота': (t1 - t0) * s0t,
        'Структура (сдвиг оборота между группами)': (gb['trx_sum'] * s0).sum() - t1 * s0t,
        'Ставки внутри групп': (gb['trx_sum'] * (s1 - s0)).sum(),
    }
    fact = gb['margin'].sum() - ga['margin'].sum()
    eff['Маржа договоров без оборота'] = fact - sum(eff.values())
    eff['ИТОГО: изменение маржи на операциях'] = fact
    eff = pd.Series(eff) / MLN
    seg = pd.DataFrame({
        'Оборот ' + m0 + ', млн': ga['trx_sum'] / MLN,
        'Оборот ' + m1 + ', млн': gb['trx_sum'] / MLN,
        'Доля оборота ' + m0 + ', %': ga['trx_sum'] / t0 * 100,
        'Доля оборота ' + m1 + ', %': gb['trx_sum'] / t1 * 100,
        'Ставка комиссии ' + m0 + ', %': ga['ops'] / ga['trx_sum'].where(ga['trx_sum'] > 0) * 100,
        'Ставка комиссии ' + m1 + ', %': gb['ops'] / gb['trx_sum'].where(gb['trx_sum'] > 0) * 100,
        'Ставка межбанка ' + m0 + ', %': ga['int_'] / ga['trx_sum'].where(ga['trx_sum'] > 0) * 100,
        'Ставка межбанка ' + m1 + ', %': gb['int_'] / gb['trx_sum'].where(gb['trx_sum'] > 0) * 100,
        'Вклад ставок, млн': gb['trx_sum'] * (s1 - s0) / MLN,
        'Изм. маржи, млн': (gb['margin'] - ga['margin']) / MLN,
    })
    return eff, seg.sort_values('Изм. маржи, млн')


tariff_eff, tariff_seg = mix_bridge(BASE_MONTH, CMP_MONTH, 'tariff')
print(f'Маржа на операциях {BASE_MONTH} -> {CMP_MONTH}, млн:')
display(tariff_eff.round(2).to_frame('млн'))
print('По тарифам:')
display(tariff_seg.round(3))

tar_share = df[df['m'].isin(CLOSED)].pivot_table(index='m', columns='tariff', values='trx_sum', aggfunc='sum')
tar_share = tar_share.div(tar_share.sum(axis=1), axis=0) * 100
tar_spread = pnl(df[df['m'].isin(CLOSED)], ['m', 'tariff'])['spread_pct'].unstack('tariff')
print('Доля оборота по тарифам, % (по месяцам):')
display(tar_share.round(1))
print('Спред (комиссия + межбанк) по тарифам, % от оборота:')
display(tar_spread.round(3))

## 8. Филиалы

Изменение Фин.реза по филиалам (млн ₽), от самого большого падения. Если падение у большинства филиалов — причина общая; если у 2–3 — смотрим их клиентов в разделе 6 (колонка «Филиал»).

In [ ]:
f0 = pnl(df[df['m'] == BASE_MONTH], 'filial')
f1 = pnl(df[df['m'] == CMP_MONTH], 'filial')
fil = pd.DataFrame({
    'Оборот ' + BASE_MONTH: f0['trx_sum'], 'Оборот ' + CMP_MONTH: f1['trx_sum'],
    'Фин.рез ' + BASE_MONTH: f0['fin'], 'Фин.рез ' + CMP_MONTH: f1['fin'],
    'Изм. оборота': f1['trx_sum'].sub(f0['trx_sum'], fill_value=0),
    'Изм. маржи на операциях': f1['margin'].sub(f0['margin'], fill_value=0),
    'Изм. ежемес. платы': f1['monthly'].sub(f0['monthly'], fill_value=0),
    'Изм. расходов': -(f1['aur'] + f1['amort']).sub(f0['aur'] + f0['amort'], fill_value=0),
    'Изм. Фин.реза': f1['fin'].sub(f0['fin'], fill_value=0),
}).fillna(0) / MLN
fil_tbl = fil.sort_values('Изм. Фин.реза')
N_FIL_DOWN = int((fil_tbl['Изм. Фин.реза'] < 0).sum())
N_FIL_TRX_DOWN = int((fil_tbl['Изм. оборота'] < 0).sum())
print(f'Фин.рез снизился у {N_FIL_DOWN} из {len(fil_tbl)} филиалов, оборот — у {N_FIL_TRX_DOWN}')
display(fil_tbl.round(2))

## 9. Ежемесячная плата: почему август выше

Первая таблица — ежемесячная плата по месяцам и источнику (`oper_doc_net` — найден документ оплаты в ЦФТ, `oper_doc_missing` — документа нет).

Вторая — договоры, у которых в `ANOMALY_MONTH` плата больше их обычной (медиана по остальным закрытым месяцам). Если сумма превышения ≈ разнице августа с соседними месяцами, значит август завышен разовыми платежами (доплата за прошлые периоды, годовая плата и т. п.), а не «потерей» в сентябре.

In [ ]:
mf = df.groupby(['m', 'monthly_src']).agg(
    sum_mln=('commission_monthly', lambda s: s.sum() / MLN),
    agreements=('agr_id', 'nunique'),
).unstack('monthly_src')
print('Ежемесячная плата по месяцам и источнику:')
display(mf.round(2))

piv = df[df['m'].isin(CLOSED)].pivot_table(index=['client', 'agr_id'], columns='m', values='commission_monthly',
                                           aggfunc='sum', fill_value=0)
others = [c for c in piv.columns if c != ANOMALY_MONTH]
piv['median_other'] = piv[others].median(axis=1)
piv['excess'] = piv[ANOMALY_MONTH] - piv['median_other']
names = df.groupby('client')['company_name'].last()
fils = df.groupby('client')['filial'].last()

m_idx = CLOSED.index(ANOMALY_MONTH)
neighbours = [CLOSED[i] for i in (m_idx - 1, m_idx + 1) if 0 <= i < len(CLOSED)]
ANOMALY_GAP = (monthly.loc[ANOMALY_MONTH, 'monthly'] - monthly.loc[neighbours, 'monthly'].mean()) / MLN
EXCESS_POS = piv.loc[piv['excess'] > 0, 'excess'].sum() / MLN
top_ex = piv.sort_values('excess', ascending=False).head(TOP_N)
EXCESS_TOP = top_ex['excess'].sum() / MLN
anomaly_top = top_ex[[c for c in CLOSED if c in top_ex.columns] + ['median_other', 'excess']].reset_index()
anomaly_top.insert(1, 'company', anomaly_top['client'].map(names))
anomaly_top.insert(2, 'filial', anomaly_top['client'].map(fils))
print(f'Ежемесячная плата {ANOMALY_MONTH} выше среднего соседних месяцев {neighbours} на {ANOMALY_GAP:,.2f} млн')
print(f'Превышение над обычной платой договоров: всего {EXCESS_POS:,.2f} млн, топ-{TOP_N} договоров — {EXCESS_TOP:,.2f} млн')
print('Договоры с наибольшим превышением (₽):')
display(anomaly_top.round(0))

## 10. Платёжные системы и ставка межбанка

Доли оборота по платёжным системам (из таблицы типов карт) рядом со ставкой межбанка из основной таблицы. Межбанк зависит от того, чьими картами платят: если доля карт других банков / дорогих систем растёт, ставка межбанка растёт.

Последняя строка — корреляция долей со ставкой межбанка по месяцам. Это подсказка, а не доказательство: месяцев мало (10).

In [ ]:
card = month_sum(CARD_TABLE, ['trx_sum'], ['payment_system'])
ps = card[card['m'].isin(CLOSED)].pivot_table(index='m', columns='payment_system', values='trx_sum', aggfunc='sum')
ps_share = ps.div(ps.sum(axis=1), axis=0) * 100
ps_share.columns = ['Доля ' + str(c) + ', %' for c in ps_share.columns]
ps_tbl = ps_share.join(pd.DataFrame({
    'Ставка межбанка, % (модуль)': -monthly['int_pct'],
    'Ставка комиссии, %': monthly['ops_pct'],
    'Спред, %': monthly['spread_pct'],
}))
corr = ps_tbl.corr()['Ставка межбанка, % (модуль)']
display(ps_tbl.round(3))
print('Корреляция со ставкой межбанка:')
display(corr.round(2).to_frame('corr'))

## 11. Сверка с Excel-отчётами коллег

Ищет в `DATA_DIR` (по умолчанию `/home/jovyan/documents/Equaring/Data`) файлы вида `09_Сентябрь_2026.xlsx` — первые две цифры = номер месяца. Строку заголовка и колонки находит сам (ID договора, ИНН, Сумма операций, ЧОД, АУР, Амортизация, Фин. Рез. и др. — названия как в прошлых сверках). Строки без ID договора (например, «Итого») исключаются, чтобы не задвоить суммы.

**Таблица 1 — какие файлы нашлись** и какие колонки в них распознаны. Если файла за новый месяц нет или колонка не нашлась — пришлите скриншот (и название файла / колонки).

**Таблица 2 — итоги месяца: DRP против Excel, разница в %.** Смотрите не на ноль, а на **стабильность**: если в сентябре–октябре разница такая же, как в январе–августе, наша методика не поменялась, и падение — не артефакт DAG.

**Таблица 3 — по договорам** (новые месяцы и сравниваемые): сколько договоров есть только у коллег / только у нас, и сколько Фин.реза на них приходится. Ниже — топ договоров с наибольшей разницей Фин.реза.

**Главная строка** — падение Фин.реза между `BASE_MONTH` и `CMP_MONTH` по отчётам коллег и по DRP. Если оба показывают падение сопоставимого размера, вывод подтверждён двумя независимыми расчётами.

In [ ]:
DATA_DIR = Path('/home/jovyan/documents/Equaring/Data')
if not DATA_DIR.exists():
    DATA_DIR = Path.home() / 'documents' / 'Equaring' / 'Data'
EXCEL_YEAR = BASE_MONTH[:4]
NEW_MONTHS = [x for x in months if x >= '2026-09']
TOL_PCT = 1.0

EX_ALIASES = {
    'agr_id': ['ID договора', 'ИД договора', 'agr_id', 'abs_agr_id'],
    'inn': ['ИНН', 'INN'],
    'trx_cnt': ['Количество операций', 'Количеств операций', 'Кол-во операций', 'trx_cnt'],
    'trx_sum': ['Сумма операций', 'Сумма опреаций', 'Оборот', 'trx_sum'],
    'commission_from_ops': ['Комиссия (% с операций)', 'Комиссия %', 'Комиссия с оборота', 'commission_from_ops'],
    'commission_monthly': ['Комиссия (₽ в месяц)', 'Комиссия (руб в месяц)', 'Комиссия руб/мес', 'Фикс комиссия',
                           'commission_monthly'],
    'int_component': ['Комиссия МПС (IRF, ₽)', 'Комиссия МПС (IRF, руб)', 'Инт компонент', 'Интерчейндж', 'int_component'],
    'chod': ['ЧОД', 'CHOD', 'chod'],
    'aur': ['АУР', 'AUR', 'aur'],
    'amortization': ['Амортизация', 'Амортизация терминалов', 'amortization'],
    'fin_result': ['Фин. Рез.', 'Фин.Рез.', 'Фин.Рез', 'Фин.рез.', 'Фин. рез.', 'Фин рез', 'Финрез',
                   'Финансовый результат', 'fin_result', 'Fin.Res.'],
    'term_cnt': ['Количество терминалов', 'Кол-во терминалов', 'term_cnt'],
}
METRIC_RU = {'trx_cnt': 'Операций', 'trx_sum': 'Оборот', 'commission_from_ops': 'Комиссия с операций',
             'commission_monthly': 'Ежемесячная плата', 'int_component': 'Межбанк', 'chod': 'ЧОД', 'aur': 'АУР',
             'amortization': 'Амортизация', 'fin_result': 'Фин.рез', 'term_cnt': 'Терминалов'}
METRICS = list(METRIC_RU)


def norm_col(c):
    return ''.join(str(c).lower().split()).rstrip('.')


ALIAS_KEYS = {k: {norm_col(x) for x in v} for k, v in EX_ALIASES.items()}


def norm_key(v):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return None
    s = str(v).strip().replace(chr(160), '').replace(' ', '')
    if s.endswith('.0'):
        s = s[:-2]
    return None if s in ('', 'nan', 'None', 'NaN') else s


def norm_inn(v):
    s = norm_key(v)
    if s is None:
        return None
    s = ''.join(ch for ch in s if ch.isdigit())
    if len(s) in (9, 11):
        s = s.zfill(len(s) + 1)
    return s or None


def to_num(s):
    t = s.astype(str).str.replace(chr(160), '', regex=False).str.replace(' ', '', regex=False)
    return pd.to_numeric(t.str.replace(',', '.', regex=False), errors='coerce')


def read_report(path):
    probe = pd.read_excel(path, header=None, nrows=15, dtype=object)
    best_i, best_n = 0, -1
    for i in range(len(probe)):
        cells = {norm_col(x) for x in probe.iloc[i].tolist() if pd.notna(x)}
        n = sum(1 for keys in ALIAS_KEYS.values() if cells & keys)
        if n > best_n:
            best_i, best_n = i, n
    ex = pd.read_excel(path, header=best_i, dtype=object)
    cols = {}
    for k, keys in ALIAS_KEYS.items():
        hit = [c for c in ex.columns if norm_col(c) in keys]
        if hit:
            cols[k] = hit[0]
    return ex, cols, best_i


EXCEL_FILES = {}
if DATA_DIR.exists():
    for p in sorted(DATA_DIR.glob('*_' + EXCEL_YEAR + '.xlsx')):
        head = p.name.split('_')[0]
        if head.isdigit() and 1 <= int(head) <= 12:
            EXCEL_FILES[EXCEL_YEAR + '-' + head.zfill(2)] = p
print('Папка с отчётами:', DATA_DIR, '| найдена:', DATA_DIR.exists())
for mth in months:
    print('  ', mth, '->', EXCEL_FILES[mth].name if mth in EXCEL_FILES else 'НЕТ ФАЙЛА')

ex_frames, info = {}, []
for mth, p in EXCEL_FILES.items():
    if mth not in months:
        continue
    try:
        ex, cols, hdr = read_report(p)
    except Exception as exc:
        info.append({'m': mth, 'Файл': p.name, 'Статус': 'ошибка чтения: ' + type(exc).__name__ + ' ' + str(exc)[:100]})
        continue
    out = pd.DataFrame(index=ex.index)
    for k, c in cols.items():
        if k == 'agr_id':
            out[k] = ex[c].map(norm_key)
        elif k == 'inn':
            out[k] = ex[c].map(norm_inn)
        else:
            out[k] = to_num(ex[c])
    no_agr = 0
    if 'agr_id' in out.columns:
        no_agr = int(out['agr_id'].isna().sum())
        out = out[out['agr_id'].notna()]
    ex_frames[mth] = out
    info.append({'m': mth, 'Файл': p.name, 'Статус': 'ok', 'Строка заголовка': hdr, 'Строк с договором': len(out),
                 'Строк без ID договора (исключены)': no_agr,
                 'Нашлись колонки': ', '.join(k for k in EX_ALIASES if k in cols),
                 'Не нашлись': ', '.join(k for k in EX_ALIASES if k not in cols)})
excel_files_tbl = pd.DataFrame(info)
print('Таблица 1. Файлы коллег:')
display(excel_files_tbl)

drp_tot = df.groupby('m')[METRICS].sum()
drp_agr = df.groupby('m')['agr_id'].nunique()
rows = []
for mth, ex in ex_frames.items():
    if 'agr_id' in ex.columns:
        rows.append({'m': mth, 'Показатель': 'Договоров', 'Excel': float(ex['agr_id'].nunique()),
                     'DRP': float(drp_agr.get(mth, np.nan)), 'Примечание': ''})
    for k in METRICS:
        if k not in ex.columns:
            continue
        e = float(ex[k].fillna(0).sum())
        d = float(drp_tot.loc[mth, k])
        note = ''
        if k == 'int_component' and e * d < 0:
            e, note = -e, 'в Excel межбанк с обратным знаком, сравнили по модулю'
        rows.append({'m': mth, 'Показатель': METRIC_RU[k], 'Excel': e, 'DRP': d, 'Примечание': note})
excel_cmp = pd.DataFrame(rows, columns=['m', 'Показатель', 'Excel', 'DRP', 'Примечание'])
excel_cmp['Разница DRP − Excel'] = excel_cmp['DRP'] - excel_cmp['Excel']
excel_cmp['Разница, %'] = excel_cmp['Разница DRP − Excel'] / excel_cmp['Excel'].abs().replace(0, np.nan) * 100
if len(excel_cmp):
    print('Таблица 2. Итоги месяца: разница DRP − Excel, %:')
    display(excel_cmp.pivot_table(index='m', columns='Показатель', values='Разница, %', aggfunc='first').round(2))
    print('Те же итоги в рублях:')
    display(excel_cmp.round(2))

names_by_agr = df.assign(agr_key=df['agr_id'].map(norm_key)).groupby('agr_key')['company_name'].last()
TOP_KEEP = ['trx_sum_ex', 'trx_sum_drp', 'chod_ex', 'chod_drp', 'commission_monthly_ex', 'commission_monthly_drp',
            'aur_ex', 'aur_drp', 'amortization_ex', 'amortization_drp', 'fin_result_ex', 'fin_result_drp', 'd_fin']
detail_rows, top_list = [], []
for mth in sorted(ex_frames):
    ex = ex_frames[mth]
    if 'agr_id' not in ex.columns or not (mth >= '2026-09' or mth in (BASE_MONTH, CMP_MONTH)):
        continue
    mets = [k for k in METRICS if k in ex.columns]
    e = ex.groupby('agr_id')[mets].sum()
    dd = df[df['m'] == mth].copy()
    dd['agr_key'] = dd['agr_id'].map(norm_key)
    d = dd.groupby('agr_key')[mets].sum()
    j = e.join(d, how='outer', lsuffix='_ex', rsuffix='_drp')
    in_e, in_d = j.index.isin(e.index), j.index.isin(d.index)
    j['where'] = np.select([in_e & in_d, in_e], ['в обоих', 'только в Excel'], default='только в DRP')
    for k in mets:
        j[k + '_ex'] = j[k + '_ex'].fillna(0)
        j[k + '_drp'] = j[k + '_drp'].fillna(0)
    row = {'m': mth}
    for w in ['в обоих', 'только в Excel', 'только в DRP']:
        row['Договоров: ' + w] = int((j['where'] == w).sum())
    if 'fin_result' in mets:
        row['Фин.рез только в Excel, млн'] = j.loc[j['where'] == 'только в Excel', 'fin_result_ex'].sum() / MLN
        row['Фин.рез только в DRP, млн'] = j.loc[j['where'] == 'только в DRP', 'fin_result_drp'].sum() / MLN
        both_agr = j[j['where'] == 'в обоих']
        dfin = both_agr['fin_result_drp'] - both_agr['fin_result_ex']
        row['Общих договоров с разницей Фин.реза > 1 ₽'] = int((dfin.abs() > 1).sum())
        row['Сумма разниц Фин.реза в общих, млн'] = dfin.sum() / MLN
        t = both_agr.assign(d_fin=dfin).reindex(dfin.abs().sort_values(ascending=False).index).head(TOP_N)
        t = t[[c for c in TOP_KEEP if c in t.columns]].copy()
        t.insert(0, 'Клиент', t.index.map(names_by_agr))
        t.insert(0, 'm', mth)
        t.index.name = 'ID договора'
        top_list.append(t.reset_index())
    detail_rows.append(row)
excel_detail = pd.DataFrame(detail_rows)
excel_top = pd.concat(top_list, ignore_index=True) if top_list else pd.DataFrame()
if len(excel_detail):
    print('Таблица 3. Сверка по договорам:')
    display(excel_detail.round(2))
    print(f'Топ-{TOP_N} общих договоров по разнице Фин.реза в каждом месяце (₽):')
    display(excel_top.round(0))

EXCEL_LINES = []
missing_new = [x for x in NEW_MONTHS if x not in ex_frames]
if missing_new:
    EXCEL_LINES.append('  Нет отчётов коллег за: ' + ', '.join(missing_new) + ' (искали в ' + str(DATA_DIR) + ')')
for mth in sorted(ex_frames):
    sub = excel_cmp[excel_cmp['m'] == mth].set_index('Показатель')
    parts, flags = [], []
    for lab in ['Фин.рез', 'ЧОД', 'Оборот', 'Договоров']:
        if lab in sub.index and pd.notna(sub.at[lab, 'Разница, %']):
            v = sub.at[lab, 'Разница, %']
            parts.append(lab + f' {v:+.2f}%')
            if lab in ('Фин.рез', 'ЧОД'):
                flags.append(abs(v) <= TOL_PCT)
    mark = '  [OK] ' if flags and all(flags) else '  [!!] '
    EXCEL_LINES.append(mark + mth + ': разница DRP − Excel: ' + (', '.join(parts) if parts else 'нет общих колонок'))
if all(x in ex_frames and 'fin_result' in ex_frames[x].columns for x in (BASE_MONTH, CMP_MONTH)):
    ex_drop = (ex_frames[CMP_MONTH]['fin_result'].sum() - ex_frames[BASE_MONTH]['fin_result'].sum()) / MLN
    drp_drop = (monthly.loc[CMP_MONTH, 'fin'] - monthly.loc[BASE_MONTH, 'fin']) / MLN
    EXCEL_LINES.append(f'  Изменение Фин.реза {BASE_MONTH} -> {CMP_MONTH}: по отчётам коллег {ex_drop:+,.2f} млн, по DRP {drp_drop:+,.2f} млн')
else:
    EXCEL_LINES.append(f'  Сравнить падение {BASE_MONTH} -> {CMP_MONTH} с отчётами коллег нельзя: нет файла или колонки Фин.реза')
print()
print('ВЫВОД ПО СВЕРКЕ С EXCEL (допуск по Фин.резу и ЧОД ±' + str(TOL_PCT) + '%):')
for s in EXCEL_LINES:
    print(s)

## 12. Итоговая справка и Excel

Справка собирается из цифр выше. Блок «ПРОВЕРКИ» — если везде `[OK]`, цифрам можно доверять: формулы сходятся, данные не потеряны, падение оборота подтверждает независимый источник, факторы объясняют изменение полностью. Если где-то `[!!]` — сначала разберитесь с этим пунктом (пришлите скриншот).

Excel со всеми таблицами — в `Data/fin_drop_analysis/`, его можно отправить коллегам.

In [ ]:
fact = (b.fin - a.fin) / MLN
resid = main_bridge['Влияние, млн'].iloc[-1]
L = []
L.append('ПРОВЕРКИ ДОСТОВЕРНОСТИ')
L.append(('  [OK] ' if FORMULA_OK else '  [!!] ') + 'Формулы ЧОД и Фин.реза сходятся построчно в сравниваемых месяцах (раздел 2)')
L.append(('  [OK] ' if abs(AGR_CHG) < 3 else '  [!!] ') + f'Число договоров {BASE_MONTH} -> {CMP_MONTH}: {AGR_CHG:+.1f}% (раздел 3)')
L.append(('  [OK] ' if CROSS_DEV < 3 else '  [!!] ') + f'Оборот подтверждён независимой таблицей MCC: там {MCC_TRX_CHG:+.1f}%, у нас {FIN_TRX_CHG:+.1f}%, отклонение пропорции {CROSS_DEV:.1f}% (раздел 3)')
L.append(('  [OK] ' if abs(resid) < 0.05 else '  [!!] ') + f'Факторы объясняют изменение полностью, невязка {resid:.3f} млн (раздел 5)')
L.append('')
L.append(f'ФИН.РЕЗ {BASE_MONTH}: {a.fin / MLN:,.2f} млн -> {CMP_MONTH}: {b.fin / MLN:,.2f} млн, изменение {fact:+,.2f} млн')
L.append(f'ЧОД {a.chod / MLN:,.2f} -> {b.chod / MLN:,.2f} млн; расходы (АУР + амортизация) {(a.aur + a.amort) / MLN:,.2f} -> {(b.aur + b.amort) / MLN:,.2f} млн')
L.append('')
L.append('ИЗ ЧЕГО СЛОЖИЛОСЬ ИЗМЕНЕНИЕ (раздел 5), от самого сильного:')
fac = main_bridge.head(len(FACTORS)).copy()
fac = fac.reindex(fac['Влияние, млн'].abs().sort_values(ascending=False).index)
for _, r in fac.iterrows():
    L.append(f'  {r.iloc[0]}: {r.iloc[1]:+,.2f} млн')
L.append(f'  Оборот {a.trx_sum / MLN:,.0f} -> {b.trx_sum / MLN:,.0f} млн ({FIN_TRX_CHG:+.1f}%); ставка комиссии {a.ops_pct:.3f}% -> {b.ops_pct:.3f}%; межбанк {-a.int_pct:.3f}% -> {-b.int_pct:.3f}%')
L.append('')
L.append('ТО ЖЕ ДЛЯ ДРУГИХ ПАР МЕСЯЦЕВ (раздел 5), факт изменения Фин.реза:')
for col in pairs_tbl.columns:
    L.append('  ' + col + ': ' + f'{pairs_tbl[col].iloc[-2]:+,.2f} млн')
L.append('')
L.append(f'КЛИЕНТЫ (раздел 6), {BASE_MONTH} -> {CMP_MONTH}:')
for name, row in cb_status.drop(index='ИТОГО').iterrows():
    L.append(f'  {name}: {int(row.iloc[0])} клиентов, оборот {row.iloc[1]:+,.1f} млн, Фин.рез {row.iloc[5]:+,.2f} млн')
L.append(f'  Оборот снизился у {SHARE_DOWN:.0f}% клиентов, работающих в обоих месяцах')
L.append(f'  Топ-10 клиентов по падению: {TOP10_DOWN_SUM:+,.2f} млн, топ-{TOP_N}: {TOP_DOWN_SUM:+,.2f} млн (из {fact:+,.2f})')
L.append('')
L.append('МАРЖА НА ОПЕРАЦИЯХ ПО ТАРИФАМ (раздел 7):')
for k, v in tariff_eff.items():
    L.append(f'  {k}: {v:+,.2f} млн')
L.append('')
L.append(f'ФИЛИАЛЫ (раздел 8): Фин.рез снизился у {N_FIL_DOWN} из {len(fil_tbl)}, оборот — у {N_FIL_TRX_DOWN}')
L.append('')
L.append(f'ЕЖЕМЕСЯЧНАЯ ПЛАТА (раздел 9): {ANOMALY_MONTH} выше соседних месяцев на {ANOMALY_GAP:,.2f} млн;')
L.append(f'  превышение над обычной платой договоров {EXCESS_POS:,.2f} млн, из них топ-{TOP_N} договоров {EXCESS_TOP:,.2f} млн')
L.append('')
L.append('СВЕРКА С EXCEL-ОТЧЁТАМИ КОЛЛЕГ (раздел 11):')
L.extend(EXCEL_LINES if EXCEL_LINES else ['  отчёты коллег не найдены'])
L.append('')
L.append(f'ВНИМАНИЕ: {OPEN_MONTH} не закрыт — ежемесячная плата за него ещё не списана, в выводы его не брать.')
for s in L:
    print(s)

out_path = OUT_DIR / ('fin_drop_' + BASE_MONTH + '_vs_' + CMP_MONTH + '.xlsx')
sheets = {
    'Справка': pd.DataFrame({'Справка': L}),
    '2 Формулы': formula, '3 Полнота': comp, '3 Сверка с MCC': cross,
    '4 PnL по месяцам': show, '4 Сезонность': season,
    '5 Факторы': main_bridge, '5 Факторы пары': pairs_tbl,
    '6 Группы клиентов': cb_status, '6 Топ падение': cb_top_down, '6 Топ рост': cb_top_up,
    '6 Топ падение оборота': cb_top_trx,
    '7 Тарифы факторы': tariff_eff.to_frame('млн'), '7 Тарифы': tariff_seg,
    '7 Доли тарифов': tar_share, '7 Спред тарифов': tar_spread,
    '8 Филиалы': fil_tbl, '9 Ежемес плата': mf, '9 Аномалия плата': anomaly_top,
    '10 Платежные системы': ps_tbl,
    '11 Excel файлы': excel_files_tbl, '11 Excel итоги': excel_cmp,
    '11 Excel договоры': excel_detail, '11 Excel топ расхождений': excel_top,
}
try:
    with pd.ExcelWriter(out_path) as xw:
        for name, t in sheets.items():
            t.to_excel(xw, sheet_name=name[:31])
    print()
    print('Excel сохранён:', out_path.resolve())
except Exception as exc:
    print('Excel не записан:', type(exc).__name__, exc)
    for name, t in sheets.items():
        t.to_csv(OUT_DIR / (name.replace(' ', '_') + '.csv'), encoding='utf-8-sig')
    print('Вместо Excel сохранены csv в', OUT_DIR.resolve())